# 01 — Exploratory Data Analysis
### NutriFit-AI · AI-Based Personalised Nutrition & Meal Recommendation System

This notebook produces the **Dataset Review / EDA evidence** for the dissertation:
distributions, correlations, missing-value analysis and data-quality findings for
both source datasets.

**Before running:** upload the whole `NutriFit-AI` folder to the root of your
Google Drive, and place the two Kaggle CSVs in `NutriFit-AI/data/raw/`
(see `data/raw/README.md` for links).

**Runtime:** `Runtime → Change runtime type → CPU`. Nothing in this notebook uses a GPU.

In [ ]:
# ============================================================
# SETUP - run this first in every notebook
# ============================================================
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/NutriFit-AI")
    if not PROJECT.exists():
        raise FileNotFoundError(
            f"{PROJECT} not found.\n"
            "Upload the whole NutriFit-AI folder to the ROOT of your Google Drive "
            "(My Drive/NutriFit-AI), then re-run this cell."
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "scikit-learn>=1.4", "pandas>=2.1", "joblib>=1.3", "seaborn>=0.13"],
        check=False,
    )
else:
    PROJECT = Path.cwd()
    while not (PROJECT / "ml" / "nutrifit").exists() and PROJECT != PROJECT.parent:
        PROJECT = PROJECT.parent

sys.path.insert(0, str(PROJECT / "ml"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nutrifit
from nutrifit import config, data, foods, labels, nutrition, planner, preprocessing, recommender, training

for directory in (config.PROCESSED_DIR, config.ARTIFACTS_DIR, config.FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print(f"nutrifit  {nutrifit.__version__}")
print(f"project   {PROJECT}")
print(f"data/raw  {config.RAW_DIR}")
print(f"figures   {config.FIGURES_DIR}")
print(f"in colab  {IN_COLAB}")

In [ ]:
def savefig(name):
    """Save the current figure into reports/figures/ for the dissertation."""
    path = config.FIGURES_DIR / f"{name}.png"
    plt.savefig(path)
    print(f"saved {path}")

## 1. Optional — download the datasets with the Kaggle API

Skip this cell if you already put the CSVs in `data/raw/`.

In [ ]:
# Uncomment to download inside Colab. You will be asked to upload kaggle.json
# (Kaggle -> Settings -> API -> Create New Token).
#
# from google.colab import files
# files.upload()                      # choose kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d valakhorasani/gym-members-exercise-dataset -p "{config.RAW_DIR}" --unzip
# !kaggle datasets download -d adilshamim8/daily-food-and-nutrition-dataset -p "{config.RAW_DIR}" --unzip
print("Files currently in data/raw:")
for f in sorted(config.RAW_DIR.glob("*.csv")):
    print(" ", f.name, f"({f.stat().st_size/1024:.0f} KB)")

## 2. Load the gym members dataset

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)-8s %(message)s")

USE_DEMO = False   # set True to explore without the real download (NOT reportable)

if USE_DEMO:
    from nutrifit.demo import make_demo_gym_dataset
    from nutrifit.schema import GYM_ALIASES, GYM_REQUIRED, normalise_columns
    gym = normalise_columns(make_demo_gym_dataset(), GYM_ALIASES, GYM_REQUIRED, "demo")
    gym["height_cm"] = gym["height_m"] * 100
    gym["bmi"] = gym["weight_kg"] / gym["height_m"] ** 2
    print("*** SYNTHETIC DEMO DATA - results are not reportable ***")
else:
    gym = data.load_gym_members()

print(f"\nShape: {gym.shape}")
gym.head()

### 2.1 Structure, dtypes and missing values

In [ ]:
summary = pd.DataFrame({
    "dtype": gym.dtypes.astype(str),
    "non_null": gym.notna().sum(),
    "missing": gym.isna().sum(),
    "missing_%": (gym.isna().mean() * 100).round(2),
    "unique": gym.nunique(),
})
display(summary)
print(f"Total missing cells: {int(gym.isna().sum().sum())}")
print(f"Duplicate rows: {int(gym.duplicated().sum())}")

### 2.2 Descriptive statistics

In [ ]:
numeric_cols = ["age", "height_cm", "weight_kg", "bmi", "fat_percentage",
                "workout_frequency", "session_duration_h", "experience_level"]
numeric_cols = [c for c in numeric_cols if c in gym.columns]
display(gym[numeric_cols].describe().T.round(2))

### 2.3 Data-quality check — supplied vs recomputed BMI

The file ships a `BMI` column. We recompute it from height and weight so the
feature is guaranteed internally consistent, and report any discrepancy.

In [ ]:
if "bmi_supplied" in gym.columns:
    diff = (gym["bmi_supplied"] - gym["bmi"]).abs()
    print(f"max abs difference : {diff.max():.4f}")
    print(f"mean abs difference: {diff.mean():.4f}")
    print(f"rows differing >0.1: {int((diff > 0.1).sum())} of {len(gym)}")
else:
    print("No supplied BMI column to compare against.")

### 2.4 Distributions of the model input features

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, column in zip(axes.ravel(), numeric_cols):
    sns.histplot(gym[column].dropna(), kde=True, ax=ax, bins=25)
    ax.set_title(column)
    ax.set_xlabel("")
for ax in axes.ravel()[len(numeric_cols):]:
    ax.set_visible(False)
fig.suptitle("Distribution of model input features", fontsize=14, y=1.02)
plt.tight_layout()
savefig("eda_feature_distributions")
plt.show()

### 2.5 Categorical breakdowns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.countplot(data=gym, x="gender", ax=axes[0]); axes[0].set_title("Gender")
sns.countplot(data=gym, x="experience_level", ax=axes[1]); axes[1].set_title("Experience level")
if "workout_type" in gym.columns:
    sns.countplot(data=gym, y="workout_type", ax=axes[2]); axes[2].set_title("Workout type")
else:
    axes[2].set_visible(False)
plt.tight_layout()
savefig("eda_categorical_counts")
plt.show()

### 2.6 Correlation structure

This is where the multiplicative structure of the problem starts to show:
`weight` and `bmi` are strongly correlated, and both drive energy requirements.

In [ ]:
plt.figure(figsize=(9, 7))
corr = gym[numeric_cols].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Feature correlation matrix (Pearson)")
savefig("eda_correlation_matrix")
plt.show()

### 2.7 Body composition by sex

Body-fat norms are sex-specific — this justifies the sex-specific reference
points used in the goal-assignment model (`nutrifit/labels.py`).

In [ ]:
if "fat_percentage" in gym.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.boxplot(data=gym, x="gender", y="fat_percentage", ax=axes[0])
    axes[0].set_title("Body fat % by sex")
    sns.scatterplot(data=gym, x="bmi", y="fat_percentage", hue="gender", alpha=0.6, ax=axes[1])
    axes[1].set_title("Body fat % vs BMI")
    plt.tight_layout()
    savefig("eda_body_composition")
    plt.show()
    display(gym.groupby("gender")["fat_percentage"].describe().round(2))

### 2.8 Derived activity bands

The dataset has no lifestyle-activity field, so the FAO/WHO activity band is
derived from weekly training volume (`frequency × session duration`).

In [ ]:
gym["_weekly_hours"] = gym["workout_frequency"] * gym["session_duration_h"]
gym["_activity"] = nutrition.derive_activity_level(gym["workout_frequency"], gym["session_duration_h"])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(gym["_weekly_hours"], bins=25, kde=True, ax=axes[0])
for cut in (1.5, 3.0, 5.0, 7.0):
    axes[0].axvline(cut, color="crimson", ls="--", lw=1)
axes[0].set_title("Weekly training hours (red = activity band cut-points)")

order = [a for a in nutrition.ACTIVITY_LEVELS if a in set(gym["_activity"])]
sns.countplot(data=gym, x="_activity", order=order, ax=axes[1])
axes[1].set_title("Derived activity band")
plt.tight_layout()
savefig("eda_activity_bands")
plt.show()

display(gym["_activity"].value_counts().rename("count").to_frame())

## 3. Food dataset

In [ ]:
if USE_DEMO:
    from nutrifit.demo import make_demo_food_dataset
    from nutrifit.schema import FOOD_ALIASES, FOOD_REQUIRED
    food_log = normalise_columns(make_demo_food_dataset(), FOOD_ALIASES, FOOD_REQUIRED, "demo")
    food_log["meal_type"] = food_log["meal_type"].str.lower()
else:
    food_log = data.load_food_dataset()

print(f"Shape: {food_log.shape}")
print(f"Distinct foods: {food_log['food_item'].nunique()}")
print(f"Meal types: {sorted(food_log['meal_type'].dropna().unique())}")
food_log.head()

### 3.1 Why the log must be aggregated into a catalogue

The file is a *consumption log*: the same food appears many times with
different portion sizes. Used row-wise it would let popular foods dominate the
recommender's candidate pool and expose portion noise as menu variety.

In [ ]:
counts = food_log["food_item"].value_counts()
print(f"Rows                     : {len(food_log)}")
print(f"Distinct foods           : {counts.size}")
print(f"Median entries per food  : {counts.median():.0f}")
print(f"Most-logged food         : {counts.index[0]} ({counts.iloc[0]} entries)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
counts.head(15).plot(kind="barh", ax=axes[0])
axes[0].invert_yaxis(); axes[0].set_title("15 most-logged foods"); axes[0].set_xlabel("log entries")

example = counts.index[0]
sns.histplot(food_log.loc[food_log["food_item"] == example, "calories"], bins=20, ax=axes[1])
axes[1].set_title(f"Portion variation for '{example}'"); axes[1].set_xlabel("kcal per log entry")
plt.tight_layout()
savefig("eda_food_log_structure")
plt.show()

### 3.2 Macro distributions and meal-slot coverage

In [ ]:
macro_cols = [c for c in ["calories", "protein_g", "carbs_g", "fat_g", "fiber_g"] if c in food_log.columns]
display(food_log[macro_cols].describe().T.round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sns.countplot(data=food_log, x="meal_type",
              order=[m for m in nutrition.MEAL_SLOTS if m in set(food_log["meal_type"])], ax=axes[0])
axes[0].set_title("Log entries per meal slot")
sns.boxplot(data=food_log, x="meal_type", y="calories",
            order=[m for m in nutrition.MEAL_SLOTS if m in set(food_log["meal_type"])], ax=axes[1])
axes[1].set_title("Calories per entry by meal slot")
plt.tight_layout()
savefig("eda_food_meal_slots")
plt.show()

### 3.3 Catalogue coverage check

The eight-week planner enforces a five-day no-repeat rule and composes each
slot from up to three items. That needs roughly **20+ items per slot** to be
satisfiable. This cell tells you whether the Kaggle catalogue alone is enough
or whether you should add the USDA bulk download.

In [ ]:
catalogue = foods.build_catalogue(food_log, verbose=False)
health = foods.catalogue_health_report(catalogue)
display(health)

thin = health[health["items"] < 20]["meal_type"].tolist()
if thin:
    print(f"\nWARNING: slots with <20 items: {thin}")
    print("Consider adding USDA FoodData Central (see data/raw/README.md section 3),")
    print("or lower min_observations in prepare_data.py to widen the pool.")
else:
    print("\nCatalogue is large enough for the 5-day variety rule.")

## 4. EDA findings to carry into the report

Write these up in the Dataset Review chapter:

1. **Row count and completeness** — 973 records, missing-value profile above.
2. **Supplied vs recomputed BMI** — quantified in §2.3; recomputing guarantees
   internal consistency between `bmi`, `height_cm` and `weight_kg`.
3. **Body composition is sex-dependent** (§2.7) — justifies sex-specific
   reference points in goal assignment.
4. **No ready-made nutrition target exists** in any public dataset — this is
   why labels are constructed from established formulas (notebook 02).
5. **The food file is a log, not a catalogue** (§3.1) — aggregation is a
   necessary preprocessing step, not an optional one.
6. **Catalogue breadth** (§3.3) — determines whether the variety constraint in
   the eight-week planner is satisfiable; drives the decision on USDA enrichment.

**Next:** `02_preprocessing.ipynb`